In [1]:
from data_generator import get_spark, generate_large_table
from pyspark.sql import functions as F
import time

# Test 1: Default GC (Parallel GC)

In [2]:
spark1=get_spark(
    "Case12_DefaultGC",{
        "spark.executor.memeory":"4g",
        "spark.executor.extraJavaOptions": "-XX:+UseParallelGC"
    }
)

26/04/18 11:15:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
df1=generate_large_table(spark1,num_rows=5_000_000)

In [4]:
start=time.time()
result1=df1.groupBy("account_number").agg(F.sum("download_bytes")).count()
print(f"Parallel GC: {time.time() - start:.1f}s")
spark1.stop()

Parallel GC: 5.7s


# Test 2: G1GC (Recommended for Spark)

In [5]:
spark2=get_spark("Case12_G1GC",{
    "spark.executor.memory": "4g",
    "spark.executor.extraJavaOptions": 
    "-XX:+UseG1GC -XX:MaxGCPauseMillis=200 -XX:+UseCompressedOops"
})
df2=generate_large_table(spark2, num_rows=5_000_000)

26/04/18 11:24:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
start=time.time()
result2=df2.groupBy("account_number").agg(F.sum("download_bytes")).count()
print(f"G1GC: {time.time() - start:.1f}s")
spark2.stop()

G1GC: 3.0s
